In [2]:
import pandas as pd 
import numpy as np


df = pd.read_csv('../data/raw/salary_benchmark.csv')


df_copy = df.copy()

df_copy = df_copy.drop(columns=['employee_id']).drop_duplicates()

In [3]:

#copy paste from 01_eda.ipynb
fix_cols = ['job_title', 'education_level', 'seniority_level']
for col in fix_cols:
    df[col] = df[col].str.strip().str.title()

value_map = {
    "Ml Engineer": "ML Engineer",
    "Hr Specialist": "HR Specialist",
    "Bachelor'S": "Bachelor",
    "Bachelors": "Bachelor",
    "Master'S": "Master",
    "Masters": "Master",
    "Mba": "MBA", 
    "Associate'S": "Associate",
    "Ph.D.": "PhD",
    "Phd": "PhD",
    "Doctorate": "PhD",
    "Mid-Level": "Mid"
}

for col in fix_cols:
    df[col] = df[col].replace(value_map)

## Impute

In [4]:
new_hire = df['years_at_company'] < 1 
df_copy.loc[new_hire & df['performance_rating'].isnull(), 'performance_rating'] = 3.0
df_copy['performance_rating'] = df_copy['performance_rating'].fillna(df_copy['performance_rating'].median())

df_copy['num_certifications'] = df_copy.groupby('job_title')['num_certifications'].transform(
    lambda x: x.fillna(x.median())
)

df_copy['years_experience'] = df_copy.groupby('seniority_level')['years_experience'].transform(
    lambda x: x.fillna(x.median())
)

print("Remaining Missing Values:", df_copy.isnull().sum().sum())

Remaining Missing Values: 0


In [ ]:
invalid_experience = df_copy['years_experience'] > (df_copy['age'] - 18)
df_copy.loc[invalid_experience, 'years_experience'] = df_copy.loc[invalid_experience, 'age'] - 18

invalid_works = df_copy['years_at_company'] > df_copy['years_experience']
df_copy.loc[invalid_works, 'years_at_company'] = df_copy.loc[invalid_works, 'years_experience']

df_copy = df_copy[df_copy['base_salary'] >= 33000].copy()


print(f"Cleaned Dataset Shape: {df_copy.shape}")
# df_copy['company_size'] = df_copy['company_size'].str.strip()
print(df_copy['company_size'].value_counts())


Cleaned Dataset Shape: (5500, 19)
company_size
Large (1000-5000)     1411
Mid (200-1000)        1331
Enterprise (5000+)    1092
Small (50-200)         988
Startup (<50)          678
Name: count, dtype: int64
education_level
Bachelor's     2286
Master's       1294
Associate's     539
MBA             507
High School     434
PhD             353
BACHELOR'S       14
bachelor's       14
Master           13
phd              11
Bachelors        11
mba              10
masters           9
PHD               5
Name: count, dtype: int64


In [12]:
for col in fix_cols:
    df_copy[col] = df_copy[col].str.strip().str.title()
    print(f"Unique values in {col}: {df_copy[col].value_counts().index.tolist()}")
    
value_map = {
    "Ml Engineer": "ML Engineer",
    "Hr Specialist": "HR Specialist",
    
    "Bachelor'S": "Bachelor",
    "Bachelors": "Bachelor",
    
    "Master'S": "Master",
    "Masters": "Master",
    "Mba": "MBA", 
    
    "Associate'S": "Associate",
    
    "Ph.D.": "PhD",
    "Phd": "PhD",
    "Doctorate": "PhD",
    
    "Mid-Level": "Mid"
}

for col in fix_cols:
    df_copy[col] = df_copy[col].replace(value_map)
    
for col in fix_cols:
    col_values = df_copy[col].value_counts()
    print(f"Cleaned unique values in {col}: {col_values.index.tolist()}")

Unique values in job_title: ['Software Engineer', 'Data Analyst', 'Data Scientist', 'Ml Engineer', 'Business Analyst', 'Finance Analyst', 'Devops Engineer', 'Product Manager', 'Marketing Analyst', 'Hr Specialist']
Unique values in education_level: ["Bachelor'S", "Master'S", "Associate'S", 'Mba', 'High School', 'Phd', 'Master', 'Bachelors', 'Masters']
Unique values in seniority_level: ['Mid', 'Senior', 'Junior', 'Lead', 'Principal', 'Mid-Level']
Cleaned unique values in job_title: ['Software Engineer', 'Data Analyst', 'Data Scientist', 'ML Engineer', 'Business Analyst', 'Finance Analyst', 'Devops Engineer', 'Product Manager', 'Marketing Analyst', 'HR Specialist']
Cleaned unique values in education_level: ['Bachelor', 'Master', 'Associate', 'MBA', 'High School', 'PhD']
Cleaned unique values in seniority_level: ['Mid', 'Senior', 'Junior', 'Lead', 'Principal']


In [14]:
print(f"Cleaned Dataset Shape: {df_copy.shape}")
# df_copy['company_size'] = df_copy['company_size'].str.strip()
print(df_copy['company_size'].value_counts())
display(df_copy['education_level'].value_counts())

Cleaned Dataset Shape: (5500, 19)
company_size
Large (1000-5000)     1411
Mid (200-1000)        1331
Enterprise (5000+)    1092
Small (50-200)         988
Startup (<50)          678
Name: count, dtype: int64


education_level
Bachelor       2325
Master         1316
Associate       539
MBA             517
High School     434
PhD             369
Name: count, dtype: int64

In [ ]:
seniority_map = {'Junior': 1, 'Mid': 2, 'Senior': 3, 'Lead': 4, 'Principal': 5}
df_copy['seniority_ordinal'] = df_copy['seniority_level'].map(seniority_map)

# Map Education
education_map = {
    'High School': 1, 
    'Associate': 2, 
    'Bachelor': 3, 
    'Master': 4, 
    'MBA': 5, 
    'PhD': 6
}
df_copy['education_ordinal'] = df_copy['education_level'].map(education_map)

size_map = {
    'Startup (<50)': 1,
    'Small (50-200)': 2,
    'Mid (200-1000)': 3,
    'Large (1000-5000)': 4,
    'Enterprise (5000+)': 5
}
if 'company_size' in df_copy.columns:
    # Just in case there's an issue with leading spaces from earlier
    df_copy['company_size'] = df_copy['company_size'].str.strip()
    df_copy['company_size_ordinal'] = df_copy['company_size'].map(size_map)

# display(df_copy['seniority_ordinal'].value_counts().sum())
# display(df_copy['education_ordinal'].value_counts().sum())